# First binary AI/non-AI baseline
## Steps 1–4 — preparation, validation, and frozen held-out test

**Current stage: test the approved C=10, balanced model at the fixed 0.5 AI cutoff.
No test-based tuning or model export.**

We predict `ai` versus `non_ai` from **description text only** using frozen
`binary-ai-v4` partitions. Enterprise labels describe what the supplied text supports;
AIToolBuzz entries are positives under the project's operational definition.
Neither establishes current product truth or employee AI-feature use.

Review stages:
1. **Reviewed:** verify inputs, inspect support, and preview training descriptions.
2. **Reviewed:** always-AI dummy and a fixed keyword rule on validation.
3. **Reviewed:** train-only TF-IDF + logistic regression; select settings on validation.
4. **This stage:** freeze the approved configuration and evaluate held-out test.

Run all cells from top to bottom. Setup instructions are in the adjacent README.
Earlier cells reconstruct the original validation experiment. The test stage refuses
a different selected configuration. Reproducing the same frozen test experiment is
allowed; it is not new independent evidence. Once evaluated, this test split must not
be used to choose future models, settings, rules, or thresholds.


## 1. Locate the repository and read the manifest

Works when Jupyter starts from the repository root or a directory inside it.
No dataset is rebuilt or modified. The frozen split seed is provenance, not a new split.


In [1]:
import csv
import hashlib
import json
import platform
from collections import Counter
from html import escape
from pathlib import Path

from IPython.display import HTML, Markdown, display

cwd = Path.cwd().resolve()
ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "data/binary_ai_dataset/v4/manifest.json").is_file()),
    None,
)
assert ROOT is not None, "Start Jupyter from within this repository."
DATA = ROOT / "data/binary_ai_dataset/v4"
manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
assert manifest["dataset_version"] == "binary-ai-v4"
assert manifest["feature_allowlist"] == ["description_text"]

print("Python:", platform.python_version())
print("Dataset:", manifest["dataset_version"])
print("Frozen source cutoff:", manifest["source_cutoff"])
print("Frozen split seed:", manifest["seed"])
print("Dataset directory:", DATA)


Python: 3.14.4
Dataset: binary-ai-v4
Frozen source cutoff: 28a32b79f8f4c7ab000e2e66ba7fd21202d3dd44
Frozen split seed: 20261014
Dataset directory: /home/newton/dev/Cyera-1A-ai-tool-discovery-and-risk-classification/data/binary_ai_dataset/v4


## 2. Verify frozen files and partition isolation

Check every exported file against its recorded SHA-256 before using it.
Test files are read as bytes for integrity verification only; test descriptions are
not loaded for exploration or model development. Group checks use the frozen assignment map.

These checks verify consistency with the manifest, not semantic label correctness.
The preparation README documents the full dataset tests and byte-identical replay.


In [2]:
required_outputs = {"assignments.json"} | {
    f"{split}{suffix}"
    for split in ("train", "validation", "test")
    for suffix in (".csv", ".metadata.json")
}
assert set(manifest["output_sha256"]) == required_outputs
for filename, expected in manifest["output_sha256"].items():
    actual = hashlib.sha256((DATA / filename).read_bytes()).hexdigest()
    assert actual == expected, f"Frozen file changed: {filename}"

assignments = json.loads((DATA / "assignments.json").read_text(encoding="utf-8"))
assert len(assignments) == manifest["examples"]
assignment_by_id = {row["example_id"]: row for row in assignments}
assert len(assignment_by_id) == len(assignments), "Duplicate assigned example ID"
assert {row["partition"] for row in assignments} == {"train", "validation", "test"}

group_partition = {}
for row in assignments:
    group = row["leakage_group_id"]
    assert group, "Missing leakage group"
    assert group_partition.setdefault(group, row["partition"]) == row["partition"], (
        f"Leakage group crosses splits: {group}"
    )

for split, stats in manifest["partitions"].items():
    split_assignments = [row for row in assignments if row["partition"] == split]
    assert len(split_assignments) == stats["examples"], split
    assert len({row["leakage_group_id"] for row in split_assignments}) == stats["groups"], split

print(f"PASS: {len(required_outputs)} frozen file checksums match.")
print(f"PASS: {len(assignments):,} unique examples in {len(group_partition):,} isolated groups.")


PASS: 7 frozen file checksums match.
PASS: 16,669 unique examples in 16,444 isolated groups.


## 3. Load training and validation data

CSV columns must be exactly `example_id, description_text, label`.
Keep exact text unchanged. Source/member metadata is loaded separately for audit and
later source-specific evaluation, **never as a model feature**.

We also reconcile row counts, labels, description hashes, assignments, and source-record
support against the frozen manifest. The test split stays unloaded.


In [3]:
development = {}
audit_metadata = {}

for split in ("train", "validation"):
    with (DATA / f"{split}.csv").open(encoding="utf-8", newline="") as stream:
        reader = csv.DictReader(stream)
        assert reader.fieldnames == ["example_id", "description_text", "label"], split
        rows = list(reader)

    metadata = json.loads((DATA / f"{split}.metadata.json").read_text(encoding="utf-8"))
    metadata_by_id = {row["example_id"]: row for row in metadata}
    ids = {row["example_id"] for row in rows}
    expected_ids = {
        row["example_id"] for row in assignments if row["partition"] == split
    }
    assert len(rows) == len(ids) == manifest["partitions"][split]["examples"], split
    assert len(metadata) == len(metadata_by_id) == len(rows), split
    assert ids == set(metadata_by_id) == expected_ids, split
    assert all(row["description_text"].strip() for row in rows), split
    assert all(row["label"] in {"ai", "non_ai"} for row in rows), split

    support = Counter()
    for row in rows:
        eid = row["example_id"]
        audit = metadata_by_id[eid]
        assert hashlib.sha256(row["description_text"].encode("utf-8")).hexdigest() == audit["description_sha256"], eid
        assert audit["leakage_group_id"] == assignment_by_id[eid]["leakage_group_id"], eid
        assert audit["members"], eid
        support[f"examples:{row['label']}"] += 1
        for member in audit["members"]:
            assert member["source"] in {"enterprise", "aitoolbuzz"}, eid
            assert member["label"] == row["label"], eid
            support[f"source_members:{member['source']}:{member['label']}"] += len(member["source_record_ids"])
    assert dict(support) == manifest["partitions"][split]["support"], split

    development[split] = rows
    audit_metadata[split] = metadata_by_id

# These lists establish the feature boundary for the next stage.
train_text = [row["description_text"] for row in development["train"]]
train_labels = [row["label"] for row in development["train"]]
validation_text = [row["description_text"] for row in development["validation"]]
validation_labels = [row["label"] for row in development["validation"]]

assert set(development) == {"train", "validation"}
print(f"PASS: {len(train_text):,} train and {len(validation_text):,} validation examples loaded.")
print("Model input: description_text only. Test text remains unloaded.")


PASS: 11,667 train and 2,501 validation examples loaded.
Model input: description_text only. Test text remains unloaded.


## 4. Class and source support

The following tables include test **counts from the manifest only**.

The first table counts final model examples. The second counts original source-record
members retained in the audit trail: duplicate consolidation and cross-source merging
mean these counts need not sum to the number of model examples.


In [4]:
class_table = ["| Split | AI | Non-AI | Total |", "|---|---:|---:|---:|"]
total_ai = total_non_ai = 0
for split, stats in manifest["partitions"].items():
    ai = stats["support"]["examples:ai"]
    non_ai = stats["support"]["examples:non_ai"]
    total_ai += ai
    total_non_ai += non_ai
    class_table.append(f"| {split} | {ai:,} | {non_ai:,} | {stats['examples']:,} |")
class_table.append(f"| **Total** | **{total_ai:,}** | **{total_non_ai:,}** | **{manifest['examples']:,}** |")
display(Markdown("\n".join(class_table)))

source_table = [
    "| Split | AIToolBuzz AI source records | Enterprise AI source records | Enterprise non-AI source records |",
    "|---|---:|---:|---:|",
]
for split, stats in manifest["partitions"].items():
    support = stats["support"]
    source_table.append(
        f"| {split} | {support['source_members:aitoolbuzz:ai']:,} "
        f"| {support['source_members:enterprise:ai']:,} "
        f"| {support['source_members:enterprise:non_ai']:,} |"
    )
display(Markdown("\n".join(source_table)))

ai_share = total_ai / manifest["examples"]
print(f"AI share of this frozen dataset: {ai_share:.2%}")
print("This is dataset composition, not estimated real-world AI prevalence.")
print(f"Always predicting AI would achieve {ai_share:.2%} pooled accuracy, with zero non-AI recall.")
print("This is a count-derived reference, not fitted-model or test prediction evaluation.")


| Split | AI | Non-AI | Total |
|---|---:|---:|---:|
| train | 11,377 | 290 | 11,667 |
| validation | 2,438 | 63 | 2,501 |
| test | 2,438 | 63 | 2,501 |
| **Total** | **16,253** | **416** | **16,669** |

| Split | AIToolBuzz AI source records | Enterprise AI source records | Enterprise non-AI source records |
|---|---:|---:|---:|
| train | 11,362 | 28 | 302 |
| validation | 2,435 | 6 | 64 |
| test | 2,435 | 6 | 65 |

AI share of this frozen dataset: 97.50%
This is dataset composition, not estimated real-world AI prevalence.
Always predicting AI would achieve 97.50% pooled accuracy, with zero non-AI recall.
This is a count-derived reference, not fitted-model or test prediction evaluation.


## 5. Preview training descriptions only

Display the first two training examples with each source/label membership.
These are deterministic illustrative examples, not representative random samples.
Descriptions are escaped before HTML rendering; source text is untrusted data.


In [5]:
for source, label in (("enterprise", "ai"), ("enterprise", "non_ai"), ("aitoolbuzz", "ai")):
    matches = [
        row for row in development["train"]
        if row["label"] == label
        and any(member["source"] == source
                for member in audit_metadata["train"][row["example_id"]]["members"])
    ]
    assert len(matches) >= 2, (source, label)
    display(Markdown(f"### {source} — {label}"))
    for row in matches[:2]:
        display(HTML(
            f"<p><code>{escape(row['example_id'])}</code></p>"
            f"<blockquote style='white-space: pre-wrap'>{escape(row['description_text'])}</blockquote>"
        ))


### enterprise — ai

### enterprise — non_ai

### aitoolbuzz — ai

## Step 1 reviewed

The data-loading and feature boundary were reviewed before adding the following baselines.
Continue using unchanged descriptions; source metadata is for evaluation cohorts only.
Test text remains unloaded.


## 6. Define the two simple baselines

- **Always-AI dummy:** predict `ai` for every description. Use scikit-learn's
  constant DummyClassifier, fitted on training inputs/labels only.
- **Keyword rule:** predict `ai` when a case-insensitive whole-token AI/ML acronym
  or phrase is present; otherwise predict `non_ai`.

The fixed rule looks for AI, AIOps, artificial intelligence, ML, machine learning,
deep learning, neural network(s), LLM(s), large language model(s), NLP, natural
language processing/understanding, or generative AI. Whitespace/hyphens can separate
phrase words. It does not use source membership, product-name lists, or annotations.

This list is specified before running validation; it is not tuned to validation mistakes.
**Limitations:** no negation/context handling; an AI mention can be unrelated or denied.
No keyword match is a heuristic negative, not proof of no AI. Implicit or sparse AI
descriptions can be missed. These baselines return labels, not calibrated confidence
or meaningful ranked scores; PR-AUC is deferred to the learned model.


In [6]:
import re
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

LABELS = ["non_ai", "ai"]
KEYWORD_PATTERN = (
    r"\b(?:AI|AIOps|artificial[\s-]+intelligence|ML|machine[\s-]+learning|"
    r"deep[\s-]+learning|neural[\s-]+networks?|LLMs?|"
    r"large[\s-]+language[\s-]+models?|NLP|"
    r"natural[\s-]+language[\s-]+(?:processing|understanding)|"
    r"generative[\s-]+AI)\b"
)
keyword_regex = re.compile(KEYWORD_PATTERN, re.IGNORECASE)

def keyword_predict(text):
    return "ai" if keyword_regex.search(text) else "non_ai"

def binary_metrics(truth, predictions):
    assert len(truth) == len(predictions) > 0
    assert set(truth) <= set(LABELS) and set(predictions) <= set(LABELS)
    matrix = confusion_matrix(truth, predictions, labels=LABELS)
    return {
        "accuracy": accuracy_score(truth, predictions),
        "macro_f1": f1_score(truth, predictions, labels=LABELS, average="macro", zero_division=0),
        "per_class": classification_report(
            truth, predictions, labels=LABELS, output_dict=True, zero_division=0
        ),
        "confusion": matrix.tolist(),
        "false_positives": int(matrix[0, 1]),
        "false_negatives": int(matrix[1, 0]),
    }

print("scikit-learn:", sklearn.__version__)
print("Fixed keyword pattern:", KEYWORD_PATTERN)


scikit-learn: 1.9.1
Fixed keyword pattern: \b(?:AI|AIOps|artificial[\s-]+intelligence|ML|machine[\s-]+learning|deep[\s-]+learning|neural[\s-]+networks?|LLMs?|large[\s-]+language[\s-]+models?|NLP|natural[\s-]+language[\s-]+(?:processing|understanding)|generative[\s-]+AI)\b


## 7. Check rule/metric behavior, then predict validation

Small synthetic assertions check phrase matching, case, word boundaries, known limitations,
confusion-matrix orientation, and undefined precision handling. They do not use dataset
labels to change the rule. Neither baseline reads test descriptions.


In [7]:
keyword_cases = [
    ("An AI-powered assistant.", "ai"),
    ("ARTIFICIAL INTELLIGENCE for document processing.", "ai"),
    ("Uses machine-learning and neural networks.", "ai"),
    ("Uses large language models and natural\nlanguage processing.", "ai"),
    ("AIOps monitoring with ML and NLP.", "ai"),
    ("Generative AI using LLMs.", "ai"),
    ("Email campaigns and daily reporting.", "non_ai"),
    ("Plain package manager for Kubernetes.", "non_ai"),
    ("This offering does not use AI.", "ai"),  # Deliberate no-negation limitation.
    ("Automated assistant handles tasks and talks naturally.", "non_ai"),
]
for text, expected in keyword_cases:
    assert keyword_predict(text) == expected, (text, expected)

check = binary_metrics(
    ["non_ai", "non_ai", "ai", "ai"],
    ["non_ai", "ai", "non_ai", "ai"],
)
assert check["confusion"] == [[1, 1], [1, 1]]
assert check["accuracy"] == check["macro_f1"] == 0.5
assert check["false_positives"] == check["false_negatives"] == 1
check = binary_metrics(["non_ai", "ai"], ["ai", "ai"])
assert check["per_class"]["non_ai"]["precision"] == 0
assert check["per_class"]["non_ai"]["recall"] == 0
print("PASS: keyword and metric self-checks.")

dummy = DummyClassifier(strategy="constant", constant="ai")
dummy.fit([[text] for text in train_text], train_labels)
validation_predictions = {
    "always_ai": dummy.predict([[text] for text in validation_text]).tolist(),
    "keyword": [keyword_predict(text) for text in validation_text],
}
assert set(validation_predictions["always_ai"]) == {"ai"}
assert all(len(predictions) == len(validation_labels)
           for predictions in validation_predictions.values())
assert set(development) == {"train", "validation"}
print(f"Validation predictions ready for {len(validation_labels):,} examples; no test predictions.")


PASS: keyword and metric self-checks.
Validation predictions ready for 2,501 examples; no test predictions.


## 8. Pooled validation comparison

Macro-F1 gives both classes equal weight. Report accuracy as context, not as the selection
criterion. A **false positive** means a non-AI example flagged as AI; a **false negative**
means an AI example missed.

Confusion matrices use **rows = actual**, **columns = predicted**, ordered non-AI then AI.
Precision with no predicted examples is displayed as zero (`zero_division=0`).
This comparison does not select a learned model or an operational threshold.


In [8]:
baseline_metrics = {
    name: binary_metrics(validation_labels, predictions)
    for name, predictions in validation_predictions.items()
}
dummy_metrics = baseline_metrics["always_ai"]
assert dummy_metrics["false_positives"] == validation_labels.count("non_ai")
assert dummy_metrics["false_negatives"] == 0
assert dummy_metrics["per_class"]["non_ai"]["recall"] == 0

summary = [
    "| Baseline | Accuracy | Macro-F1 | False positives | False negatives |",
    "|---|---:|---:|---:|---:|",
]
per_class = [
    "| Baseline | Class | Support | Precision | Recall | F1 |",
    "|---|---|---:|---:|---:|---:|",
]
for name, metrics in baseline_metrics.items():
    summary.append(
        f"| {name} | {metrics['accuracy']:.4f} | {metrics['macro_f1']:.4f} "
        f"| {metrics['false_positives']} | {metrics['false_negatives']} |"
    )
    for label in LABELS:
        values = metrics["per_class"][label]
        per_class.append(
            f"| {name} | {label} | {int(values['support'])} "
            f"| {values['precision']:.4f} | {values['recall']:.4f} | {values['f1-score']:.4f} |"
        )
display(Markdown("\n".join(summary)))
display(Markdown("\n".join(per_class)))

for name, metrics in baseline_metrics.items():
    matrix = metrics["confusion"]
    display(Markdown(
        f"### {name} — validation confusion matrix\n"
        "| Actual / Predicted | non_ai | ai |\n|---|---:|---:|\n"
        f"| non_ai | {matrix[0][0]} | {matrix[0][1]} |\n"
        f"| ai | {matrix[1][0]} | {matrix[1][1]} |"
    ))


| Baseline | Accuracy | Macro-F1 | False positives | False negatives |
|---|---:|---:|---:|---:|
| always_ai | 0.9748 | 0.4936 | 63 | 0 |
| keyword | 0.5090 | 0.3782 | 0 | 1228 |

| Baseline | Class | Support | Precision | Recall | F1 |
|---|---|---:|---:|---:|---:|
| always_ai | non_ai | 63 | 0.0000 | 0.0000 | 0.0000 |
| always_ai | ai | 2438 | 0.9748 | 1.0000 | 0.9872 |
| keyword | non_ai | 63 | 0.0488 | 1.0000 | 0.0931 |
| keyword | ai | 2438 | 1.0000 | 0.4963 | 0.6634 |

### always_ai — validation confusion matrix
| Actual / Predicted | non_ai | ai |
|---|---:|---:|
| non_ai | 0 | 63 |
| ai | 0 | 2438 |

### keyword — validation confusion matrix
| Actual / Predicted | non_ai | ai |
|---|---:|---:|
| non_ai | 63 | 0 |
| ai | 1228 | 1210 |

## 9. Source-specific validation behavior

Report both:
- **Enterprise-only:** examples whose final source is `enterprise`, excluding cross-source merges.
- **Enterprise-member:** examples with any enterprise member, including cross-source merges.
  Each model example is counted once, regardless of its number of source records.

For **AIToolBuzz-member** examples, report only AI recall/misses because that source has
no negatives. Merged examples participate in both member cohorts, so these cohorts overlap
and must not be summed. The pure enterprise cohort and member cohort may have different
AI support. Small enterprise-AI supports make recall noisy.


In [9]:
validation_rows = development["validation"]
validation_audit = audit_metadata["validation"]
enterprise_only_indices = [
    i for i, row in enumerate(validation_rows)
    if validation_audit[row["example_id"]]["source"] == "enterprise"
]
member_indices = {
    source: [
        i for i, row in enumerate(validation_rows)
        if any(member["source"] == source
               for member in validation_audit[row["example_id"]]["members"])
    ]
    for source in ("enterprise", "aitoolbuzz")
}
assert set(enterprise_only_indices) <= set(member_indices["enterprise"])
assert set(member_indices["enterprise"]) | set(member_indices["aitoolbuzz"]) == set(range(len(validation_rows)))

enterprise_results = {}
for cohort, indices in (
    ("enterprise-only", enterprise_only_indices),
    ("enterprise-member", member_indices["enterprise"]),
):
    truth = [validation_labels[i] for i in indices]
    assert set(truth) == set(LABELS), f"Expected both classes in {cohort}"
    enterprise_results[cohort] = {}
    display(Markdown(f"### {cohort} — {len(indices)} model examples"))
    table = [
        "| Baseline | Class | Support | Precision | Recall | F1 |",
        "|---|---|---:|---:|---:|---:|",
    ]
    for name, predictions in validation_predictions.items():
        metrics = binary_metrics(truth, [predictions[i] for i in indices])
        enterprise_results[cohort][name] = metrics
        for label in LABELS:
            values = metrics["per_class"][label]
            table.append(
                f"| {name} | {label} | {int(values['support'])} "
                f"| {values['precision']:.4f} | {values['recall']:.4f} | {values['f1-score']:.4f} |"
            )
        print(
            f"{cohort} / {name}: accuracy={metrics['accuracy']:.4f}, "
            f"macro-F1={metrics['macro_f1']:.4f}, "
            f"false positives={metrics['false_positives']}, false negatives={metrics['false_negatives']}"
        )
    display(Markdown("\n".join(table)))

catalog_indices = member_indices["aitoolbuzz"]
assert catalog_indices and all(validation_labels[i] == "ai" for i in catalog_indices)
catalog_table = [
    "| Baseline | AIToolBuzz-member AI examples | Correct AI | Missed AI | AI recall |",
    "|---|---:|---:|---:|---:|",
]
for name, predictions in validation_predictions.items():
    correct = sum(predictions[i] == "ai" for i in catalog_indices)
    missed = len(catalog_indices) - correct
    catalog_table.append(
        f"| {name} | {len(catalog_indices)} | {correct} | {missed} | {correct / len(catalog_indices):.4f} |"
    )
display(Markdown("\n".join(catalog_table)))


### enterprise-only — 69 model examples

enterprise-only / always_ai: accuracy=0.0870, macro-F1=0.0800, false positives=63, false negatives=0
enterprise-only / keyword: accuracy=1.0000, macro-F1=1.0000, false positives=0, false negatives=0


| Baseline | Class | Support | Precision | Recall | F1 |
|---|---|---:|---:|---:|---:|
| always_ai | non_ai | 63 | 0.0000 | 0.0000 | 0.0000 |
| always_ai | ai | 6 | 0.0870 | 1.0000 | 0.1600 |
| keyword | non_ai | 63 | 1.0000 | 1.0000 | 1.0000 |
| keyword | ai | 6 | 1.0000 | 1.0000 | 1.0000 |

### enterprise-member — 69 model examples

enterprise-member / always_ai: accuracy=0.0870, macro-F1=0.0800, false positives=63, false negatives=0


enterprise-member / keyword: accuracy=1.0000, macro-F1=1.0000, false positives=0, false negatives=0


| Baseline | Class | Support | Precision | Recall | F1 |
|---|---|---:|---:|---:|---:|
| always_ai | non_ai | 63 | 0.0000 | 0.0000 | 0.0000 |
| always_ai | ai | 6 | 0.0870 | 1.0000 | 0.1600 |
| keyword | non_ai | 63 | 1.0000 | 1.0000 | 1.0000 |
| keyword | ai | 6 | 1.0000 | 1.0000 | 1.0000 |

| Baseline | AIToolBuzz-member AI examples | Correct AI | Missed AI | AI recall |
|---|---:|---:|---:|---:|
| always_ai | 2432 | 2432 | 0 | 1.0000 |
| keyword | 2432 | 1204 | 1228 | 0.4951 |

## 10. Inspect validation mistakes

Show the first three errors of each type for each baseline, in frozen row order.
Full counts are above; these previews are not a representative error sample.
Descriptions remain unchanged and are escaped for display.
Do not silently expand the keyword list after reading mistakes: any revised rule is
a separate validation-development experiment, not an independently evaluated baseline.


In [10]:
for name, predictions in validation_predictions.items():
    for actual, predicted, error_type in (
        ("non_ai", "ai", "False positives — non-AI flagged as AI"),
        ("ai", "non_ai", "False negatives — missed AI"),
    ):
        indices = [
            i for i, label in enumerate(validation_labels)
            if label == actual and predictions[i] == predicted
        ]
        expected_count = baseline_metrics[name][
            "false_positives" if actual == "non_ai" else "false_negatives"
        ]
        assert len(indices) == expected_count
        display(Markdown(f"### {name}: {error_type} ({len(indices)} total)"))
        if not indices:
            print("None.")
        for i in indices[:3]:
            row = validation_rows[i]
            origin = validation_audit[row["example_id"]]["source"]
            match = keyword_regex.search(row["description_text"]) if name == "keyword" else None
            reason = f"; matched text: {match.group(0)}" if match else ""
            display(HTML(
                f"<p><code>{escape(row['example_id'])}</code><br>"
                f"Source: {escape(origin)}; actual: {actual}; predicted: {predicted}{escape(reason)}</p>"
                f"<blockquote style='white-space: pre-wrap'>{escape(row['description_text'])}</blockquote>"
            ))


### always_ai: False positives — non-AI flagged as AI (63 total)

### always_ai: False negatives — missed AI (0 total)

None.


### keyword: False positives — non-AI flagged as AI (0 total)

None.


### keyword: False negatives — missed AI (1228 total)

## Step 2 reviewed — baseline findings

### What this validation run tells us

- **Always-AI:** accuracy 97.48%, but non-AI recall is zero; macro-F1 is 0.4936.
- **Keyword:** no false positives, but 1,228 missed AI examples out of 2,438;
  AI recall is 49.63% and macro-F1 is 0.3782.
- The keyword rule gets all **69 enterprise validation examples** right
  (6 AI and 63 non-AI). This small slice is not evidence of broad enterprise generalization.
- AIToolBuzz-member AI recall is **49.51%** (1,204 of 2,432). Sparse descriptions,
  implicit AI functionality, and token-boundary restrictions contribute to misses.
  Error previews illustrate possibilities, not a measured breakdown of all causes.

The keyword rule does **not** beat the dummy on pooled macro-F1, despite correctly
recognizing every non-AI validation example. The two baselines expose different failure
modes; the learned model must be compared with both, including source-specific behavior.
The fixed rule has not been expanded after observing validation mistakes.

The user reviewed this checkpoint and authorized the learned-model comparison below.
The original keyword rule remains unchanged. The fixed test set remains reserved.

Limitations remain: source/style confounding, sparse catalog descriptions, highly
imbalanced classes, and small enterprise-AI evaluation supports. This offline result
does not establish current product truth, employee AI-feature use, runtime website-text
transfer, or universal previously-unseen Tool generalization.


## 11. Define the learned-model comparison before evaluating it

**Model:** TF-IDF word unigrams/bigrams → L2-regularized logistic regression.
Tokens must appear in at least two training descriptions (`min_df=2`); keep stop
words and otherwise use standard vectorizer defaults. A scikit-learn Pipeline
keeps fitted preprocessing and classification together.

Compare exactly six settings:
- `C ∈ {0.1, 1, 10}` (smaller C means stronger regularization).
- `class_weight=None` versus `class_weight="balanced"`.
  Balanced weighting gives minority training examples more influence without
  deleting positives or synthesizing negatives.

**Selection:** highest pooled validation macro-F1 at the default 0.5 AI score cutoff.
On an exact tie, keep the first experiment: lower C first, then ordinary weighting.
Report enterprise behavior alongside selection; it is not a hidden selection criterion.

Every pipeline fits **train text and train labels only**. Validation uses transform/
predict only. The test text remains unloaded. Use the frozen seed for reproducibility,
and treat any convergence warning as an error rather than accepting an unfinished fit.

Probability outputs are **uncalibrated model scores**, particularly with balanced class
weighting; they are not verified confidence or estimates of enterprise prevalence.
A default cutoff is not an endorsed operational decision policy.


In [11]:
import warnings
import numpy as np
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc, average_precision_score, precision_recall_curve
from sklearn.pipeline import Pipeline

pipeline_template = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("classifier", LogisticRegression(
        solver="liblinear", max_iter=1000, random_state=manifest["seed"]
    )),
])
EXPERIMENT_C = (0.1, 1.0, 10.0)
EXPERIMENT_WEIGHTS = (None, "balanced")
DEFAULT_AI_CUTOFF = 0.5

# Check train-only vocabulary/IDF and class-column handling on a tiny synthetic fit.
toy = clone(pipeline_template)
toy.fit(
    ["conventional records conventional", "conventional reports conventional",
     "neural learned neural", "neural models neural"],
    ["non_ai", "non_ai", "ai", "ai"],
)
toy_vectorizer = toy.named_steps["tfidf"]
toy_vocab_before = dict(toy_vectorizer.vocabulary_)
toy_idf_before = toy_vectorizer.idf_.copy()
toy_scores = toy.predict_proba(["validationonlytoken neural"])
assert "validationonlytoken" not in toy_vectorizer.vocabulary_
assert toy_vectorizer.vocabulary_ == toy_vocab_before
assert np.array_equal(toy_vectorizer.idf_, toy_idf_before)
assert set(toy.classes_) == set(LABELS)
assert np.allclose(toy_scores.sum(axis=1), 1)
assert np.isfinite(toy_scores).all()
assert toy_vectorizer.transform(["validationonlytoken"]).nnz == 0
assert average_precision_score([0, 1], [0.1, 0.9]) == 1
precision, recall, _ = precision_recall_curve([0, 1], [0.1, 0.9])
assert auc(recall, precision) == 1
print("PASS: synthetic train-only preprocessing, probability, and ranking checks.")


PASS: synthetic train-only preprocessing, probability, and ranking checks.


## 12. Fit six train-only pipelines and select by validation macro-F1

The table exposes the minority-class tradeoff and enterprise AI hits as well as the
selection metric. Enterprise hits use the enterprise-member cohort defined earlier.
Models stay in notebook memory; no files or datasets are written by these cells.


In [12]:
experiments = []
for C in EXPERIMENT_C:
    for class_weight in EXPERIMENT_WEIGHTS:
        name = f"C={C:g}, weight={class_weight or 'ordinary'}"
        pipeline = clone(pipeline_template).set_params(
            classifier__C=C, classifier__class_weight=class_weight
        )
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            pipeline.fit(train_text, train_labels)

        classifier = pipeline.named_steps["classifier"]
        assert set(classifier.classes_) == set(LABELS)
        ai_column = classifier.classes_.tolist().index("ai")
        vectorizer = pipeline.named_steps["tfidf"]
        vocab_before = dict(vectorizer.vocabulary_)
        idf_before = vectorizer.idf_.copy()

        predictions = pipeline.predict(validation_text).tolist()
        ai_scores = pipeline.predict_proba(validation_text)[:, ai_column]
        assert len(predictions) == len(ai_scores) == len(validation_labels)
        assert np.isfinite(ai_scores).all() and ((ai_scores >= 0) & (ai_scores <= 1)).all()
        assert predictions == np.where(ai_scores >= DEFAULT_AI_CUTOFF, "ai", "non_ai").tolist()
        assert vectorizer.vocabulary_ == vocab_before
        assert np.array_equal(vectorizer.idf_, idf_before)
        assert (classifier.n_iter_ < classifier.max_iter).all()

        experiments.append({
            "name": name, "C": C, "class_weight": class_weight,
            "pipeline": pipeline, "predictions": predictions, "ai_scores": ai_scores,
            "metrics": binary_metrics(validation_labels, predictions),
        })

assert len(experiments) == 6
selected_experiment = max(experiments, key=lambda item: item["metrics"]["macro_f1"])
selected_pipeline = selected_experiment["pipeline"]
selected_predictions = selected_experiment["predictions"]
selected_ai_scores = selected_experiment["ai_scores"]
selected_metrics = selected_experiment["metrics"]

grid_table = [
    "| Setting | Macro-F1 | Non-AI F1 | False positives | Missed AI | Enterprise AI hits |",
    "|---|---:|---:|---:|---:|---:|",
]
enterprise_ai_indices = [
    i for i in member_indices["enterprise"] if validation_labels[i] == "ai"
]
for experiment in experiments:
    metrics = experiment["metrics"]
    hits = sum(experiment["predictions"][i] == "ai" for i in enterprise_ai_indices)
    setting = experiment["name"] + (" **← selected**" if experiment is selected_experiment else "")
    grid_table.append(
        f"| {setting} | {metrics['macro_f1']:.4f} "
        f"| {metrics['per_class']['non_ai']['f1-score']:.4f} "
        f"| {metrics['false_positives']} | {metrics['false_negatives']} "
        f"| {hits}/{len(enterprise_ai_indices)} |"
    )
display(Markdown("\n".join(grid_table)))
print("Selected by pooled validation macro-F1:", selected_experiment["name"])
print("Training vocabulary size:", len(selected_pipeline.named_steps["tfidf"].vocabulary_))
print("Validation transforms/predictions left vocabulary and IDF unchanged.")


| Setting | Macro-F1 | Non-AI F1 | False positives | Missed AI | Enterprise AI hits |
|---|---:|---:|---:|---:|---:|
| C=0.1, weight=ordinary | 0.4936 | 0.0000 | 63 | 0 | 6/6 |
| C=0.1, weight=balanced | 0.8071 | 0.6272 | 10 | 53 | 2/6 |
| C=1, weight=ordinary | 0.6316 | 0.2740 | 53 | 0 | 6/6 |
| C=1, weight=balanced | 0.8713 | 0.7500 | 9 | 27 | 3/6 |
| C=10, weight=ordinary | 0.8699 | 0.7451 | 25 | 1 | 6/6 |
| C=10, weight=balanced **← selected** | 0.9152 | 0.8346 | 10 | 11 | 4/6 |

Selected by pooled validation macro-F1: C=10, weight=balanced
Training vocabulary size: 18926
Validation transforms/predictions left vocabulary and IDF unchanged.


## 13. Compare the selected model with both simple baselines

This is validation-development evidence, not final test performance.
Report both classes, confusion counts, and ranked-score metrics.

For the learned model, show **trapezoidal precision–recall AUC** and **average precision
(AP)** separately; they are not the same calculation. AP for constant scores equals
positive-class prevalence. The AI class is so common that a large AI score alone
can be misleading; inspect non-AI metrics too.


In [13]:
comparison_predictions = {**validation_predictions, "tfidf_lr": selected_predictions}
comparison_metrics = {**baseline_metrics, "tfidf_lr": selected_metrics}
table = [
    "| Model | Accuracy | Macro-F1 | False positives | Missed AI |",
    "|---|---:|---:|---:|---:|",
]
for name, metrics in comparison_metrics.items():
    table.append(
        f"| {name} | {metrics['accuracy']:.4f} | {metrics['macro_f1']:.4f} "
        f"| {metrics['false_positives']} | {metrics['false_negatives']} |"
    )
display(Markdown("\n".join(table)))

table = ["| Class | Support | Precision | Recall | F1 |", "|---|---:|---:|---:|---:|"]
for label in LABELS:
    values = selected_metrics["per_class"][label]
    table.append(
        f"| {label} | {int(values['support'])} | {values['precision']:.4f} "
        f"| {values['recall']:.4f} | {values['f1-score']:.4f} |"
    )
display(Markdown("\n".join(table)))
matrix = selected_metrics["confusion"]
display(Markdown(
    "### Selected model — validation confusion matrix\n"
    "| Actual / Predicted | non_ai | ai |\n|---|---:|---:|\n"
    f"| non_ai | {matrix[0][0]} | {matrix[0][1]} |\n"
    f"| ai | {matrix[1][0]} | {matrix[1][1]} |"
))

ranking_metrics = {}
table = [
    "| Positive class | PR-AUC (trapezoidal) | Average precision | Constant-score AP reference |",
    "|---|---:|---:|---:|",
]
for label, scores in (("ai", selected_ai_scores), ("non_ai", 1 - selected_ai_scores)):
    truth = np.array(validation_labels) == label
    precision, recall, _ = precision_recall_curve(truth, scores)
    ranking_metrics[label] = {
        "pr_auc": auc(recall, precision),
        "average_precision": average_precision_score(truth, scores),
        "constant_ap": float(truth.mean()),
    }
    values = ranking_metrics[label]
    table.append(
        f"| {label} | {values['pr_auc']:.4f} | {values['average_precision']:.4f} "
        f"| {values['constant_ap']:.4f} |"
    )
display(Markdown("\n".join(table)))

for name, metrics in baseline_metrics.items():
    delta = selected_metrics["macro_f1"] - metrics["macro_f1"]
    print(f"Validation macro-F1 difference versus {name}: {delta:+.4f}")
zero_vector_count = sum(selected_pipeline.named_steps["tfidf"].transform(validation_text).getnnz(axis=1) == 0)
print(f"Validation descriptions with no retained vocabulary tokens: {zero_vector_count}")
print("No-token inputs get intercept-only scores; this stage does not implement abstention.")


| Model | Accuracy | Macro-F1 | False positives | Missed AI |
|---|---:|---:|---:|---:|
| always_ai | 0.9748 | 0.4936 | 63 | 0 |
| keyword | 0.5090 | 0.3782 | 0 | 1228 |
| tfidf_lr | 0.9916 | 0.9152 | 10 | 11 |

| Class | Support | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| non_ai | 63 | 0.8281 | 0.8413 | 0.8346 |
| ai | 2438 | 0.9959 | 0.9955 | 0.9957 |

### Selected model — validation confusion matrix
| Actual / Predicted | non_ai | ai |
|---|---:|---:|
| non_ai | 53 | 10 |
| ai | 11 | 2427 |

| Positive class | PR-AUC (trapezoidal) | Average precision | Constant-score AP reference |
|---|---:|---:|---:|
| ai | 0.9998 | 0.9998 | 0.9748 |
| non_ai | 0.8868 | 0.8872 | 0.0252 |

Validation macro-F1 difference versus always_ai: +0.4215
Validation macro-F1 difference versus keyword: +0.5370
Validation descriptions with no retained vocabulary tokens: 0
No-token inputs get intercept-only scores; this stage does not implement abstention.


## 14. Check enterprise transfer and catalog recall

Reuse the same cohort definitions and example-level counting as Step 2.
Report all three models together. Source metadata selects evaluation rows only;
it was not passed to TF-IDF or logistic regression.


In [14]:
selected_enterprise_metrics = {}
for cohort, indices in (
    ("enterprise-only", enterprise_only_indices),
    ("enterprise-member", member_indices["enterprise"]),
):
    truth = [validation_labels[i] for i in indices]
    learned = binary_metrics(truth, [selected_predictions[i] for i in indices])
    selected_enterprise_metrics[cohort] = learned
    all_metrics = {**enterprise_results[cohort], "tfidf_lr": learned}
    display(Markdown(f"### {cohort} — {len(indices)} model examples"))
    table = [
        "| Model | Macro-F1 | Non-AI recall | AI recall | False positives | Missed AI |",
        "|---|---:|---:|---:|---:|---:|",
    ]
    for name, metrics in all_metrics.items():
        table.append(
            f"| {name} | {metrics['macro_f1']:.4f} "
            f"| {metrics['per_class']['non_ai']['recall']:.4f} "
            f"| {metrics['per_class']['ai']['recall']:.4f} "
            f"| {metrics['false_positives']} | {metrics['false_negatives']} |"
        )
    display(Markdown("\n".join(table)))
    print(
        f"Selected model: enterprise non-AI support={truth.count('non_ai')}, "
        f"enterprise AI support={truth.count('ai')}"
    )
    print(classification_report(
        truth, [selected_predictions[i] for i in indices], labels=LABELS,
        digits=4, zero_division=0
    ))

table = [
    "| Model | AIToolBuzz-member AI examples | Correct AI | Missed AI | AI recall |",
    "|---|---:|---:|---:|---:|",
]
for name, predictions in comparison_predictions.items():
    correct = sum(predictions[i] == "ai" for i in catalog_indices)
    table.append(
        f"| {name} | {len(catalog_indices)} | {correct} "
        f"| {len(catalog_indices) - correct} | {correct / len(catalog_indices):.4f} |"
    )
display(Markdown("\n".join(table)))


### enterprise-only — 69 model examples

| Model | Macro-F1 | Non-AI recall | AI recall | False positives | Missed AI |
|---|---:|---:|---:|---:|---:|
| always_ai | 0.0800 | 0.0000 | 1.0000 | 63 | 0 |
| keyword | 1.0000 | 1.0000 | 1.0000 | 0 | 0 |
| tfidf_lr | 0.6492 | 0.8413 | 0.6667 | 10 | 2 |

Selected model: enterprise non-AI support=63, enterprise AI support=6
              precision    recall  f1-score   support

      non_ai     0.9636    0.8413    0.8983        63
          ai     0.2857    0.6667    0.4000         6

    accuracy                         0.8261        69
   macro avg     0.6247    0.7540    0.6492        69
weighted avg     0.9047    0.8261    0.8550        69



### enterprise-member — 69 model examples

| Model | Macro-F1 | Non-AI recall | AI recall | False positives | Missed AI |
|---|---:|---:|---:|---:|---:|
| always_ai | 0.0800 | 0.0000 | 1.0000 | 63 | 0 |
| keyword | 1.0000 | 1.0000 | 1.0000 | 0 | 0 |
| tfidf_lr | 0.6492 | 0.8413 | 0.6667 | 10 | 2 |

Selected model: enterprise non-AI support=63, enterprise AI support=6
              precision    recall  f1-score   support

      non_ai     0.9636    0.8413    0.8983        63
          ai     0.2857    0.6667    0.4000         6

    accuracy                         0.8261        69
   macro avg     0.6247    0.7540    0.6492        69
weighted avg     0.9047    0.8261    0.8550        69



| Model | AIToolBuzz-member AI examples | Correct AI | Missed AI | AI recall |
|---|---:|---:|---:|---:|
| always_ai | 2432 | 2432 | 0 | 1.0000 |
| keyword | 2432 | 1204 | 1228 | 0.4951 |
| tfidf_lr | 2432 | 2423 | 9 | 0.9963 |

## 15. Inspect learned terms and validation mistakes

Show ten largest AI-associated and non-AI-associated coefficients. These describe
the fitted linear model, not causal explanations or proof of AI functionality.
Embedded product/vendor names are still part of description text; excluding metadata
does not prevent source/style shortcuts.

Show the first three mistakes of each type, plus enterprise AI misses, in frozen
validation order, with raw AI
scores and the original keyword decision. These previews do not measure how often a
particular failure cause occurs, and they must not change the frozen test set.


In [15]:
classifier = selected_pipeline.named_steps["classifier"]
feature_names = selected_pipeline.named_steps["tfidf"].get_feature_names_out()
# A binary coefficient favors classes_[1]; orient the display toward AI explicitly.
ai_coefficients = classifier.coef_[0] * (1 if classifier.classes_[1] == "ai" else -1)
for heading, order in (
    ("AI-associated terms", np.argsort(ai_coefficients)[-10:][::-1]),
    ("Non-AI-associated terms", np.argsort(ai_coefficients)[:10]),
):
    display(Markdown(f"### {heading}"))
    table = ["| Term | Signed AI coefficient |", "|---|---:|"]
    for i in order:
        table.append(f"| {escape(feature_names[i])} | {ai_coefficients[i]:+.4f} |")
    display(Markdown("\n".join(table)))

for actual, predicted, heading, cohort_indices, cohort_metrics in (
    ("non_ai", "ai", "False positives — non-AI flagged as AI",
     range(len(validation_labels)), selected_metrics),
    ("ai", "non_ai", "False negatives — missed AI",
     range(len(validation_labels)), selected_metrics),
    ("ai", "non_ai", "Enterprise-member AI misses",
     member_indices["enterprise"], selected_enterprise_metrics["enterprise-member"]),
):
    indices = [
        i for i in cohort_indices
        if validation_labels[i] == actual and selected_predictions[i] == predicted
    ]
    expected = cohort_metrics["false_positives" if actual == "non_ai" else "false_negatives"]
    assert len(indices) == expected
    display(Markdown(f"### Selected model: {heading} ({len(indices)} total)"))
    if not indices:
        print("None.")
    for i in indices[:3]:
        row = validation_rows[i]
        origin = validation_audit[row["example_id"]]["source"]
        display(HTML(
            f"<p><code>{escape(row['example_id'])}</code><br>"
            f"Source: {escape(origin)}; actual: {actual}; predicted: {predicted}<br>"
            f"Raw AI score: {selected_ai_scores[i]:.4f}; keyword decision: {keyword_predict(row['description_text'])}</p>"
            f"<blockquote style='white-space: pre-wrap'>{escape(row['description_text'])}</blockquote>"
        ))
assert set(development) == {"train", "validation"}
print("PASS: learned stage used description text only; test text remains unloaded.")


### AI-associated terms

| Term | Signed AI coefficient |
|---|---:|
| ai | +12.9729 |
| powered | +4.8022 |
| ai powered | +4.7064 |
| an ai | +4.6601 |
| text | +2.8983 |
| various | +2.8709 |
| learning | +2.7827 |
| businesses | +2.7092 |
| tool that | +2.6943 |
| analysis | +2.6402 |

### Non-AI-associated terms

| Term | Signed AI coefficient |
|---|---:|
| is | -10.9733 |
| the | -9.2660 |
| to | -8.2661 |
| and | -8.2472 |
| services | -7.3337 |
| of | -7.3208 |
| managed | -6.6190 |
| application | -6.2951 |
| enterprise | -6.2810 |
| software | -6.0747 |

### Selected model: False positives — non-AI flagged as AI (10 total)

### Selected model: False negatives — missed AI (11 total)

### Selected model: Enterprise-member AI misses (2 total)

PASS: learned stage used description text only; test text remains unloaded.


## Step 3 reviewed — selected validation experiment

### Selected validation experiment: C=10, balanced class weighting

- Pooled accuracy **99.16%**, macro-F1 **0.9152**, versus dummy **0.4936** and keyword **0.3782**.
- Non-AI: **53/63** correctly rejected; precision **82.81%**, recall **84.13%**, F1 **0.8346**.
- AI: **2,427/2,438** detected; precision **99.59%**, recall **99.55%**.
- Enterprise: **4/6 AI detected**, with **10/63 non-AI falsely flagged**.
  Enterprise macro-F1 is **0.6492**, below the keyword rule's **1.0000** on this small slice.
- Catalog: **2,423/2,432 AI detected** (99.63% recall).

**Interpretation:** a useful first learned benchmark, not a promoted enterprise model.
It beats both simple baselines on pooled macro-F1, but not the keyword rule on
enterprise validation. The top negative coefficients include common words such as
"is", "the", "to", and "and" as well as enterprise-software terms—consistent with a
possible writing-style shortcut, not proof of one. Metadata exclusion alone does not
remove that risk. We have not changed the grid or keyword rule after seeing results.

The user reviewed the results and approved held-out testing of this exact
configuration at the unchanged 0.5 cutoff. The next cells implement that evaluation,
not an operational-policy endorsement. Six AI enterprise validation examples cannot
support a strong transfer claim. No calibration or model export is implemented.

Good pooled validation performance does not establish enterprise generalization,
runtime website-text transfer, current product truth, or employee AI-feature use.


## 16. Freeze the approved experiment before loading test text

**Approved:** word TF-IDF unigrams/bigrams, `min_df=2`; logistic regression
`C=10`, `class_weight="balanced"`, `solver="liblinear"`, seed `20261014`;
AI when its raw model score is **≥ 0.5**.

Use the train-fitted pipeline already selected on validation. Do not refit on
train+validation, tune thresholds, change the keyword rule, or score alternative
learned settings on test. Check the selected configuration before reading test text.
Snapshot fitted preprocessing and coefficients to verify prediction does not mutate them.


In [16]:
FROZEN_TEST_POLICY = {
    "dataset_version": "binary-ai-v4",
    "C": 10.0,
    "class_weight": "balanced",
    "solver": "liblinear",
    "max_iter": 1000,
    "seed": 20261014,
    "ngram_range": (1, 2),
    "min_df": 2,
    "ai_cutoff": 0.5,
    "fit_partition": "train",
    "selection_partition": "validation",
}
assert manifest["dataset_version"] == FROZEN_TEST_POLICY["dataset_version"]
assert selected_experiment["C"] == FROZEN_TEST_POLICY["C"]
assert selected_experiment["class_weight"] == FROZEN_TEST_POLICY["class_weight"]
assert DEFAULT_AI_CUTOFF == FROZEN_TEST_POLICY["ai_cutoff"]

test_vectorizer = selected_pipeline.named_steps["tfidf"]
test_classifier = selected_pipeline.named_steps["classifier"]
for parameter in ("C", "class_weight", "solver", "max_iter"):
    assert test_classifier.get_params()[parameter] == FROZEN_TEST_POLICY[parameter], parameter
assert test_classifier.random_state == FROZEN_TEST_POLICY["seed"]
assert test_vectorizer.ngram_range == FROZEN_TEST_POLICY["ngram_range"]
assert test_vectorizer.min_df == FROZEN_TEST_POLICY["min_df"]
assert set(test_classifier.classes_) == set(LABELS)

frozen_vocab = dict(test_vectorizer.vocabulary_)
frozen_idf = test_vectorizer.idf_.copy()
frozen_coefficients = test_classifier.coef_.copy()
frozen_intercept = test_classifier.intercept_.copy()
frozen_keyword_pattern = keyword_regex.pattern
print("Frozen test policy:", FROZEN_TEST_POLICY)
print("PASS: approved model and cutoff verified before loading test text.")


Frozen test policy: {'dataset_version': 'binary-ai-v4', 'C': 10.0, 'class_weight': 'balanced', 'solver': 'liblinear', 'max_iter': 1000, 'seed': 20261014, 'ngram_range': (1, 2), 'min_df': 2, 'ai_cutoff': 0.5, 'fit_partition': 'train', 'selection_partition': 'validation'}
PASS: approved model and cutoff verified before loading test text.


## 17. Load and validate the held-out test split

This is the first modeling-stage load of test descriptions. Reconcile rows, labels,
description hashes, provenance support, and group assignments with the frozen manifest.
Test truth and source metadata are used for reporting only, never fitting or selection.


In [17]:
with (DATA / "test.csv").open(encoding="utf-8", newline="") as stream:
    reader = csv.DictReader(stream)
    assert reader.fieldnames == ["example_id", "description_text", "label"]
    test_rows = list(reader)
test_metadata = json.loads((DATA / "test.metadata.json").read_text(encoding="utf-8"))
test_audit = {row["example_id"]: row for row in test_metadata}
test_ids = {row["example_id"] for row in test_rows}
expected_test_ids = {
    row["example_id"] for row in assignments if row["partition"] == "test"
}
assert len(test_rows) == len(test_ids) == manifest["partitions"]["test"]["examples"]
assert len(test_metadata) == len(test_audit) == len(test_rows)
assert test_ids == set(test_audit) == expected_test_ids
assert test_ids.isdisjoint({row["example_id"] for rows in development.values() for row in rows})
assert all(row["description_text"].strip() and row["label"] in LABELS for row in test_rows)

support = Counter()
for row in test_rows:
    audit = test_audit[row["example_id"]]
    assert hashlib.sha256(row["description_text"].encode("utf-8")).hexdigest() == audit["description_sha256"]
    assert audit["leakage_group_id"] == assignment_by_id[row["example_id"]]["leakage_group_id"]
    assert audit["members"]
    support[f"examples:{row['label']}"] += 1
    for member in audit["members"]:
        assert member["source"] in {"enterprise", "aitoolbuzz"}
        assert member["label"] == row["label"]
        support[f"source_members:{member['source']}:{member['label']}"] += len(member["source_record_ids"])
assert dict(support) == manifest["partitions"]["test"]["support"]

test_text = [row["description_text"] for row in test_rows]
test_labels = [row["label"] for row in test_rows]
print(f"PASS: {len(test_rows):,} held-out test examples match frozen support and assignments.")


PASS: 2,501 held-out test examples match frozen support and assignments.


## 18. Predict once for the settled model and compare baselines

The dummy and keyword rule are unchanged. No alternative learned settings are evaluated.
The selected model uses its fitted text preprocessing and the fixed 0.5 cutoff.
Confusion matrices again use rows = actual, columns = predicted, ordered non-AI then AI.


In [18]:
test_ai_column = test_classifier.classes_.tolist().index("ai")
test_ai_scores = selected_pipeline.predict_proba(test_text)[:, test_ai_column]
assert len(test_ai_scores) == len(test_labels)
assert np.isfinite(test_ai_scores).all() and ((test_ai_scores >= 0) & (test_ai_scores <= 1)).all()
test_predictions = {
    "always_ai": dummy.predict([[text] for text in test_text]).tolist(),
    "keyword": [keyword_predict(text) for text in test_text],
    "tfidf_lr": np.where(test_ai_scores >= FROZEN_TEST_POLICY["ai_cutoff"], "ai", "non_ai").tolist(),
}
assert set(test_predictions["always_ai"]) == {"ai"}
assert all(len(predictions) == len(test_labels) for predictions in test_predictions.values())
assert test_predictions["tfidf_lr"] == selected_pipeline.predict(test_text).tolist()
assert test_vectorizer.vocabulary_ == frozen_vocab
assert np.array_equal(test_vectorizer.idf_, frozen_idf)
assert np.array_equal(test_classifier.coef_, frozen_coefficients)
assert np.array_equal(test_classifier.intercept_, frozen_intercept)
assert keyword_regex.pattern == frozen_keyword_pattern

test_metrics = {
    name: binary_metrics(test_labels, predictions)
    for name, predictions in test_predictions.items()
}
assert test_metrics["always_ai"]["false_positives"] == test_labels.count("non_ai")
assert test_metrics["always_ai"]["false_negatives"] == 0

table = [
    "| Test model | Accuracy | Macro-F1 | False positives | Missed AI |",
    "|---|---:|---:|---:|---:|",
]
for name, metrics in test_metrics.items():
    table.append(
        f"| {name} | {metrics['accuracy']:.4f} | {metrics['macro_f1']:.4f} "
        f"| {metrics['false_positives']} | {metrics['false_negatives']} |"
    )
display(Markdown("\n".join(table)))

table = [
    "| Test model | Class | Support | Precision | Recall | F1 |",
    "|---|---|---:|---:|---:|---:|",
]
for name, metrics in test_metrics.items():
    for label in LABELS:
        values = metrics["per_class"][label]
        table.append(
            f"| {name} | {label} | {int(values['support'])} | {values['precision']:.4f} "
            f"| {values['recall']:.4f} | {values['f1-score']:.4f} |"
        )
display(Markdown("\n".join(table)))
for name, metrics in test_metrics.items():
    matrix = metrics["confusion"]
    display(Markdown(
        f"### {name} — test confusion matrix\n"
        "| Actual / Predicted | non_ai | ai |\n|---|---:|---:|\n"
        f"| non_ai | {matrix[0][0]} | {matrix[0][1]} |\n"
        f"| ai | {matrix[1][0]} | {matrix[1][1]} |"
    ))
print("PASS: fitted vocabulary, IDF, coefficients, intercept, and keyword rule unchanged.")


| Test model | Accuracy | Macro-F1 | False positives | Missed AI |
|---|---:|---:|---:|---:|
| always_ai | 0.9748 | 0.4936 | 63 | 0 |
| keyword | 0.5154 | 0.3817 | 0 | 1212 |
| tfidf_lr | 0.9960 | 0.9593 | 5 | 5 |

| Test model | Class | Support | Precision | Recall | F1 |
|---|---|---:|---:|---:|---:|
| always_ai | non_ai | 63 | 0.0000 | 0.0000 | 0.0000 |
| always_ai | ai | 2438 | 0.9748 | 1.0000 | 0.9872 |
| keyword | non_ai | 63 | 0.0494 | 1.0000 | 0.0942 |
| keyword | ai | 2438 | 1.0000 | 0.5029 | 0.6692 |
| tfidf_lr | non_ai | 63 | 0.9206 | 0.9206 | 0.9206 |
| tfidf_lr | ai | 2438 | 0.9979 | 0.9979 | 0.9979 |

### always_ai — test confusion matrix
| Actual / Predicted | non_ai | ai |
|---|---:|---:|
| non_ai | 0 | 63 |
| ai | 0 | 2438 |

### keyword — test confusion matrix
| Actual / Predicted | non_ai | ai |
|---|---:|---:|
| non_ai | 63 | 0 |
| ai | 1212 | 1226 |

### tfidf_lr — test confusion matrix
| Actual / Predicted | non_ai | ai |
|---|---:|---:|
| non_ai | 58 | 5 |
| ai | 5 | 2433 |

PASS: fitted vocabulary, IDF, coefficients, intercept, and keyword rule unchanged.


## 19. Held-out score ranking and validation comparison

PR-AUC is trapezoidal; AP is average precision. The constant-score AP reference is
test class prevalence, not a real-world estimate. Scores remain uncalibrated.
The validation-to-test comparison describes this already-settled experiment only.


In [19]:
test_ranking_metrics = {}
table = [
    "| Positive class | Test PR-AUC (trapezoidal) | Test AP | Constant-score AP reference |",
    "|---|---:|---:|---:|",
]
for label, scores in (("ai", test_ai_scores), ("non_ai", 1 - test_ai_scores)):
    truth = np.array(test_labels) == label
    precision, recall, _ = precision_recall_curve(truth, scores)
    test_ranking_metrics[label] = {
        "pr_auc": auc(recall, precision),
        "average_precision": average_precision_score(truth, scores),
        "constant_ap": float(truth.mean()),
    }
    values = test_ranking_metrics[label]
    table.append(
        f"| {label} | {values['pr_auc']:.4f} | {values['average_precision']:.4f} "
        f"| {values['constant_ap']:.4f} |"
    )
display(Markdown("\n".join(table)))

metrics = test_metrics["tfidf_lr"]
table = [
    "| Selected model split | Accuracy | Macro-F1 | False positives | Missed AI |",
    "|---|---:|---:|---:|---:|",
]
for split, values in (("validation", selected_metrics), ("test", metrics)):
    table.append(
        f"| {split} | {values['accuracy']:.4f} | {values['macro_f1']:.4f} "
        f"| {values['false_positives']} | {values['false_negatives']} |"
    )
display(Markdown("\n".join(table)))
for name, baseline in test_metrics.items():
    if name != "tfidf_lr":
        print(f"Test macro-F1 difference versus {name}: {metrics['macro_f1'] - baseline['macro_f1']:+.4f}")
print("Test descriptions with no retained vocabulary tokens:",
      int(sum(test_vectorizer.transform(test_text).getnnz(axis=1) == 0)))


| Positive class | Test PR-AUC (trapezoidal) | Test AP | Constant-score AP reference |
|---|---:|---:|---:|
| ai | 0.9999 | 0.9999 | 0.9748 |
| non_ai | 0.8858 | 0.8876 | 0.0252 |

| Selected model split | Accuracy | Macro-F1 | False positives | Missed AI |
|---|---:|---:|---:|---:|
| validation | 0.9916 | 0.9152 | 10 | 11 |
| test | 0.9960 | 0.9593 | 5 | 5 |

Test macro-F1 difference versus always_ai: +0.4657
Test macro-F1 difference versus keyword: +0.5776
Test descriptions with no retained vocabulary tokens: 0


## 20. Enterprise and catalog test behavior

Reuse pure enterprise and enterprise-member definitions; merged/member cohorts can
overlap, and source-record counts differ from model-example counts. Show all six
enterprise AI decisions explicitly. Product names in that table are audit labels,
not model features. With six positives, one miss changes enterprise recall by 16.7 points.


In [20]:
test_enterprise_only = [
    i for i, row in enumerate(test_rows) if test_audit[row["example_id"]]["source"] == "enterprise"
]
test_member_indices = {
    source: [
        i for i, row in enumerate(test_rows)
        if any(member["source"] == source for member in test_audit[row["example_id"]]["members"])
    ]
    for source in ("enterprise", "aitoolbuzz")
}
assert set(test_enterprise_only) <= set(test_member_indices["enterprise"])
assert set(test_member_indices["enterprise"]) | set(test_member_indices["aitoolbuzz"]) == set(range(len(test_rows)))
test_enterprise_metrics = {}
for cohort, indices in (
    ("enterprise-only", test_enterprise_only),
    ("enterprise-member", test_member_indices["enterprise"]),
):
    truth = [test_labels[i] for i in indices]
    assert set(truth) == set(LABELS)
    test_enterprise_metrics[cohort] = {}
    display(Markdown(f"### {cohort} — {len(indices)} test examples ({truth.count('ai')} AI, {truth.count('non_ai')} non-AI)"))
    table = [
        "| Model | Macro-F1 | Non-AI recall | AI recall | False positives | Missed AI |",
        "|---|---:|---:|---:|---:|---:|",
    ]
    for name, predictions in test_predictions.items():
        metrics = binary_metrics(truth, [predictions[i] for i in indices])
        test_enterprise_metrics[cohort][name] = metrics
        table.append(
            f"| {name} | {metrics['macro_f1']:.4f} "
            f"| {metrics['per_class']['non_ai']['recall']:.4f} "
            f"| {metrics['per_class']['ai']['recall']:.4f} "
            f"| {metrics['false_positives']} | {metrics['false_negatives']} |"
        )
    display(Markdown("\n".join(table)))
    print(classification_report(
        truth, [test_predictions["tfidf_lr"][i] for i in indices],
        labels=LABELS, digits=4, zero_division=0
    ))

test_catalog_indices = test_member_indices["aitoolbuzz"]
assert test_catalog_indices and all(test_labels[i] == "ai" for i in test_catalog_indices)
table = [
    "| Model | AIToolBuzz-member AI examples | Correct AI | Missed AI | AI recall |",
    "|---|---:|---:|---:|---:|",
]
for name, predictions in test_predictions.items():
    correct = sum(predictions[i] == "ai" for i in test_catalog_indices)
    table.append(
        f"| {name} | {len(test_catalog_indices)} | {correct} "
        f"| {len(test_catalog_indices) - correct} | {correct / len(test_catalog_indices):.4f} |"
    )
display(Markdown("\n".join(table)))

table = [
    "| Enterprise AI product (audit metadata) | Dummy | Keyword | Learned | Raw AI score |",
    "|---|---|---|---|---:|",
]
test_enterprise_ai_indices = [
    i for i in test_member_indices["enterprise"] if test_labels[i] == "ai"
]
assert len(test_enterprise_ai_indices) == manifest["enterprise_only_test_slice"]["source_member_support"]["ai"]
for i in test_enterprise_ai_indices:
    names = sorted({
        record["product_name"]
        for member in test_audit[test_rows[i]["example_id"]]["members"]
        if member["source"] == "enterprise"
        for record in member["metadata"]["source_records"]
    })
    product = escape(", ".join(names)).replace("|", "\\|")
    table.append(
        f"| {product} | {test_predictions['always_ai'][i]} "
        f"| {test_predictions['keyword'][i]} | {test_predictions['tfidf_lr'][i]} "
        f"| {test_ai_scores[i]:.4f} |"
    )
display(Markdown("\n".join(table)))


### enterprise-only — 69 test examples (6 AI, 63 non-AI)

| Model | Macro-F1 | Non-AI recall | AI recall | False positives | Missed AI |
|---|---:|---:|---:|---:|---:|
| always_ai | 0.0800 | 0.0000 | 1.0000 | 63 | 0 |
| keyword | 1.0000 | 1.0000 | 1.0000 | 0 | 0 |
| tfidf_lr | 0.6820 | 0.9206 | 0.5000 | 5 | 3 |

              precision    recall  f1-score   support

      non_ai     0.9508    0.9206    0.9355        63
          ai     0.3750    0.5000    0.4286         6

    accuracy                         0.8841        69
   macro avg     0.6629    0.7103    0.6820        69
weighted avg     0.9007    0.8841    0.8914        69



### enterprise-member — 69 test examples (6 AI, 63 non-AI)

| Model | Macro-F1 | Non-AI recall | AI recall | False positives | Missed AI |
|---|---:|---:|---:|---:|---:|
| always_ai | 0.0800 | 0.0000 | 1.0000 | 63 | 0 |
| keyword | 1.0000 | 1.0000 | 1.0000 | 0 | 0 |
| tfidf_lr | 0.6820 | 0.9206 | 0.5000 | 5 | 3 |

              precision    recall  f1-score   support

      non_ai     0.9508    0.9206    0.9355        63
          ai     0.3750    0.5000    0.4286         6

    accuracy                         0.8841        69
   macro avg     0.6629    0.7103    0.6820        69
weighted avg     0.9007    0.8841    0.8914        69



| Model | AIToolBuzz-member AI examples | Correct AI | Missed AI | AI recall |
|---|---:|---:|---:|---:|
| always_ai | 2432 | 2432 | 0 | 1.0000 |
| keyword | 2432 | 1220 | 1212 | 0.5016 |
| tfidf_lr | 2432 | 2430 | 2 | 0.9992 |

| Enterprise AI product (audit metadata) | Dummy | Keyword | Learned | Raw AI score |
|---|---|---|---|---:|
| Comprehend | ai | ai | ai | 0.9495 |
| Datacap | ai | ai | ai | 0.9634 |
| Nexidia Analytics | ai | ai | ai | 0.5077 |
| Playbooks | ai | ai | non_ai | 0.0029 |
| Operations Management | ai | ai | non_ai | 0.0011 |
| Intelligent Customer Engagement Platform | ai | ai | non_ai | 0.0181 |

## 21. Inspect settled-model test mistakes — not for tuning

Show the first three false positives and false negatives, plus enterprise AI misses,
in frozen test order. This is diagnosis of the settled result, not permission to
modify the model/rule/cutoff and reuse test as an independent evaluation.


In [21]:
for actual, predicted, heading, indices, cohort_metrics in (
    ("non_ai", "ai", "False positives — non-AI flagged as AI",
     range(len(test_labels)), test_metrics["tfidf_lr"]),
    ("ai", "non_ai", "False negatives — missed AI",
     range(len(test_labels)), test_metrics["tfidf_lr"]),
    ("ai", "non_ai", "Enterprise-member AI misses",
     test_member_indices["enterprise"], test_enterprise_metrics["enterprise-member"]["tfidf_lr"]),
):
    errors = [
        i for i in indices if test_labels[i] == actual and test_predictions["tfidf_lr"][i] == predicted
    ]
    assert len(errors) == cohort_metrics["false_positives" if actual == "non_ai" else "false_negatives"]
    display(Markdown(f"### Selected model: {heading} ({len(errors)} total)"))
    if not errors:
        print("None.")
    for i in errors[:3]:
        row = test_rows[i]
        origin = test_audit[row["example_id"]]["source"]
        display(HTML(
            f"<p><code>{escape(row['example_id'])}</code><br>"
            f"Source: {escape(origin)}; actual: {actual}; predicted: {predicted}<br>"
            f"Raw AI score: {test_ai_scores[i]:.4f}; keyword decision: {keyword_predict(row['description_text'])}</p>"
            f"<blockquote style='white-space: pre-wrap'>{escape(row['description_text'])}</blockquote>"
        ))
print("Held-out test evaluated for the frozen experiment; no settings changed.")


### Selected model: False positives — non-AI flagged as AI (5 total)

### Selected model: False negatives — missed AI (5 total)

### Selected model: Enterprise-member AI misses (3 total)

Held-out test evaluated for the frozen experiment; no settings changed.


## Stop here — review held-out test

### Frozen C=10 / balanced / 0.5 test result

| Measure | Held-out result |
|---|---:|
| Pooled accuracy | **99.60%** |
| Pooled macro-F1 | **0.9593** |
| Non-AI correctly rejected | **58/63 (92.06%)** |
| AI detected | **2,433/2,438 (99.79%)** |
| False positives / missed AI | **5 / 5** |
| Enterprise AI detected | **3/6 (50%)** |
| Enterprise macro-F1 | **0.6820** |
| Catalog-member AI detected | **2,430/2,432 (99.92%)** |

Pooled macro-F1 is above both the always-AI dummy (**0.4936**) and fixed keyword
rule (**0.3817**), and above the selected model's validation macro-F1 (**0.9152**).
This is a strong first learned benchmark for the frozen pooled dataset.

**Do not hide the enterprise result:** the keyword rule gets the 69 enterprise test
examples right, while the learned model falsely flags 5 non-AI descriptions and
misses 3 AI descriptions. It misses Playbooks, Micro Focus Operations Management,
and the Intelligent Customer Engagement Platform despite explicit AI/ML signals.
Their low raw AI scores illustrate why scores must not be treated as verified confidence.
Small support makes recall noisy, but these concrete misses still matter.

No configuration, training data, keyword rule, or cutoff changed after evaluation.
No alternatives were selected on test. The fitted preprocessing and classifier
remained unchanged during prediction.

**The test split has now been evaluated.** Do not use these outcomes to select future
settings, thresholds, or keyword rules and then describe this split as independent
final evaluation. Re-running the identical frozen experiment only reproduces this result.

No model bundle, application integration, calibration, embeddings, category model,
new labels, or dataset changes were made. This remains an offline benchmark, not a
promoted enterprise model or proof of website-text transfer, current product truth,
or employee AI-feature use. Source/style confounding remains a material limitation.
